In [3]:
!pip install pyttsx3

  Using cached pypiwin32-223-py3-none-any.whl.metadata (236 bytes)
   ---------------------------------------- 0.0/241.5 kB ? eta -:--:--
   - -------------------------------------- 10.2/241.5 kB ? eta -:--:--
   ------ -------------------------------- 41.0/241.5 kB 487.6 kB/s eta 0:00:01
   ------ -------------------------------- 41.0/241.5 kB 487.6 kB/s eta 0:00:01
   ----------- --------------------------- 71.7/241.5 kB 435.7 kB/s eta 0:00:01
   ------------------- ------------------ 122.9/241.5 kB 552.2 kB/s eta 0:00:01
   ------------------------ ------------- 153.6/241.5 kB 654.6 kB/s eta 0:00:01
   ----------------------------- -------- 184.3/241.5 kB 654.4 kB/s eta 0:00:01
   -------------------------------------  235.5/241.5 kB 719.7 kB/s eta 0:00:01
   -------------------------------------- 241.5/241.5 kB 672.2 kB/s eta 0:00:00
Using cached pypiwin32-223-py3-none-any.whl (1.7 kB)


In [7]:
!pip install SpeechRecognition

   ---------------------------------------- 0.0/32.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/32.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/32.9 MB 495.5 kB/s eta 0:01:07
   ---------------------------------------- 0.1/32.9 MB 491.5 kB/s eta 0:01:07
   ---------------------------------------- 0.1/32.9 MB 602.4 kB/s eta 0:00:55
   ---------------------------------------- 0.2/32.9 MB 743.9 kB/s eta 0:00:44
   ---------------------------------------- 0.2/32.9 MB 765.3 kB/s eta 0:00:43
   ---------------------------------------- 0.3/32.9 MB 780.5 kB/s eta 0:00:42
   ---------------------------------------- 0.3/32.9 MB 855.7 kB/s eta 0:00:39
   ---------------------------------------- 0.4/32.9 MB 851.3 kB/s eta 0:00:39
    --------------------------------------- 0.5/32.9 MB 1.1 MB/s eta 0:00:29
    --------------------------------------- 0.7/32.9 MB 1.3 MB/s eta 0:00:25
    --------------------------------------- 0.7/32.9 MB 1.4 MB/s eta 0:00:

In [9]:
!pip install SpeechRecognition==2.1.3

   ---------------------------------------- 0.0/619.7 kB ? eta -:--:--
   - -------------------------------------- 30.7/619.7 kB 1.4 MB/s eta 0:00:01
   --- ------------------------------------ 61.4/619.7 kB 1.1 MB/s eta 0:00:01
   --- ------------------------------------ 61.4/619.7 kB 1.1 MB/s eta 0:00:01
   ------ ------------------------------- 112.6/619.7 kB 656.4 kB/s eta 0:00:01
   ----------- -------------------------- 194.6/619.7 kB 908.0 kB/s eta 0:00:01
   ----------- -------------------------- 194.6/619.7 kB 908.0 kB/s eta 0:00:01
   --------------- ---------------------- 245.8/619.7 kB 795.7 kB/s eta 0:00:01
   -------------------- ----------------- 327.7/619.7 kB 925.5 kB/s eta 0:00:01
   ---------------------------- ----------- 440.3/619.7 kB 1.2 MB/s eta 0:00:01
   -------------------------------- ------- 501.8/619.7 kB 1.2 MB/s eta 0:00:01
   ------------------------------------- -- 573.4/619.7 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 619.7/61

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from joblib import dump, load
import colorsys
import re
import pyttsx3
import speech_recognition as sr

# Initialize components
engine = pyttsx3.init()
recognizer = sr.Recognizer()

# ================== CORE SYSTEM FUNCTIONS ==================
def speak(text):
    """Convert text to speech"""
    print(f"System: {text}")
    engine.say(text)
    engine.runAndWait()

def listen():
    """Listen to user voice input"""
    with sr.Microphone() as source:
        print("Listening...")
        recognizer.adjust_for_ambient_noise(source)
        try:
            audio = recognizer.listen(source, timeout=5)
            text = recognizer.recognize_google(audio)
            print(f"User: {text}")
            return text.lower()
        except sr.UnknownValueError:
            speak("Sorry, I didn't catch that. Could you repeat?")
            return None
        except Exception as e:
            print(f"Error: {e}")
            speak("I'm having trouble with the microphone. Please check your audio settings.")
            return None

# ================== CROP RECOMMENDATION SYSTEM ==================
def load_dataset(csv_path):
    """Load and preprocess the crop recommendation dataset"""
    try:
        df = pd.read_csv(csv_path)
        
        # If dataset doesn't have RGB values directly
        if 'image_color' in df.columns:
            def extract_rgb(color_str):
                match = re.match(r'rgb\((\d+),\s*(\d+),\s*(\d+)\)', color_str)
                if match:
                    return tuple(map(int, match.groups()))
                return (0, 0, 0)

            df['R'] = df['image_color'].apply(lambda x: extract_rgb(x)[0])
            df['G'] = df['image_color'].apply(lambda x: extract_rgb(x)[1])
            df['B'] = df['image_color'].apply(lambda x: extract_rgb(x)[2])
        else:
            # Assume we have R, G, B columns directly
            if not all(col in df.columns for col in ['R', 'G', 'B']):
                raise ValueError("Dataset must contain either 'image_color' or 'R','G','B' columns")

        def get_color_features(row):
            r, g, b = row['R']/255, row['G']/255, row['B']/255
            h, l, s = colorsys.rgb_to_hls(r, g, b)
            return pd.Series([h, l, s], index=['Hue', 'Lightness', 'Saturation'])
        
        color_features = df.apply(get_color_features, axis=1)
        df = pd.concat([df, color_features], axis=1)
        
        return df
    
    except Exception as e:
        print(f"Error loading dataset: {e}")
        speak("There was a problem loading the crop database.")
        return None

def prepare_model(df):
    """Prepare the recommendation model"""
    try:
        features = ['R', 'G', 'B', 'Hue', 'Lightness', 'Saturation']
        df = df.dropna(subset=features + ['Plant_Name'])
        
        le_plant = LabelEncoder()
        y = le_plant.fit_transform(df['Plant_Name'])
        
        scaler = StandardScaler()
        X = scaler.fit_transform(df[features])
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        
        dump(model, 'plant_model.pkl')
        dump(scaler, 'soil_scaler.pkl')
        dump(le_plant, 'plant_encoder.pkl')
        
        return model, scaler, le_plant
    except Exception as e:
        print(f"Error preparing model: {e}")
        return None, None, None

# Load or train model
try:
    plant_model = load('plant_model.pkl')
    soil_scaler = load('soil_scaler.pkl')
    le_plant = load('plant_encoder.pkl')
except:
    print("Training new model...")
    df = load_dataset('final_soil_dataset.csv')
    if df is not None:
        plant_model, soil_scaler, le_plant = prepare_model(df)
    else:
        speak("System cannot start without the dataset. Please check the data file.")
        exit()

def predict_plant(R, G, B, top_n=3):
    """Predict suitable plants based on soil color"""
    try:
        r, g, b = R/255, G/255, B/255
        h, l, s = colorsys.rgb_to_hls(r, g, b)
        
        input_data = np.array([[R, G, B, h, l, s]])
        input_scaled = soil_scaler.transform(input_data)
        
        probas = plant_model.predict_proba(input_scaled)[0]
        top_idx = np.argsort(probas)[-top_n:][::-1]
        percentages = (probas[top_idx] * 100).round(1)
        plant_names = le_plant.inverse_transform(top_idx)
        
        return {
            "plants": plant_names,
            "confidences": percentages
        }
    except Exception as e:
        print(f"Prediction error: {e}")
        return None

# ================== MAIN APPLICATION ==================
def get_soil_color():
    """Ask about soil color in English"""
    color_mapping = {
        'black': {'R': 0, 'G': 0, 'B': 0},
        'red': {'R': 255, 'G': 0, 'B': 0},
        'alluvial': {'R': 210, 'G': 180, 'B': 140},
        'clay': {'R': 189, 'G': 161, 'B': 137},
    }
    
    while True:
        speak("What color is your soil? Please say: black, red, alluvial, clay")
        response = listen()
        
        if not response:
            continue
            
        for color_name, rgb_values in color_mapping.items():
            if color_name in response:
                speak(f"You selected {color_name} soil.")
                return rgb_values
        
        speak("I didn't recognize that soil color. Please try again.")

def main():
    """Main program execution"""
    try:
        speak("Welcome to EzaSavvy!")
        speak("We'll recommend crops based on your soil color.")
        
        while True:
            soil_color = get_soil_color()
            
            speak("Analyzing your soil color...")
            results = predict_plant(**soil_color)
            
            if results is None:
                speak("Sorry, I couldn't analyze that soil color.")
            else:
                speak("Here are my recommendations for your soil:")
                for i, (plant, confidence) in enumerate(zip(results['plants'], results['confidences']), 1):
                    speak(f"{i}. {plant} with {confidence:.1f}% suitability")
                
                best_plant = results['plants'][0]
                speak(f"I recommend planting {best_plant} as your primary crop.")
            
            speak("Would you like to analyze another soil? Say yes or no.")
            response = listen()
            
            if not response or 'no' in response:
                speak("Thank you for EzaSavvy. Happy farming!")
                break

    except KeyboardInterrupt:
        print("\nProgram stopped by user")
    except Exception as e:
        print(f"Error: {e}")
        speak("System error occurred. Please restart the application.")

if __name__ == "__main__":
    main()
    